<a href="https://colab.research.google.com/github/CuriousTechNomad/slm-agentic-software-engineering/blob/main/notebooks/04_multi_agent_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Phase 1: Agent Framework
Goal

Build reusable infrastructure:

Load baseline tasks
Load model
Create reusable Agent class
Create reusable inference engine
Test one agent

# Notebook 04 - Multi-Agent Workflow (Phase 1)

## Objective

Build the reusable agent framework that will be used throughout the project.

This notebook introduces:

- Generic Agent abstraction
- Prompt execution
- Model inference
- Response generation

Later phases will compose multiple agents into a complete software engineering workflow.

In [1]:
from google.colab import drive
from pathlib import Path
import os
import json

drive.mount('/content/drive')

PROJECT_DIR = Path("/content/drive/MyDrive/LLM_Project")

OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR = PROJECT_DIR / "data"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
CONFIG_DIR = PROJECT_DIR / "configs"

for folder in [
    OUTPUT_DIR,
    DATA_DIR,
    CHECKPOINT_DIR,
    CONFIG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print(PROJECT_DIR)

Mounted at /content/drive
/content/drive/MyDrive/LLM_Project


In [2]:
!pip -q install transformers accelerate pandas

In [3]:
import json
import time
import torch
import pandas as pd

from dataclasses import dataclass

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [4]:
config = {

    "model_name": "Qwen/Qwen2.5-3B-Instruct",

    "max_new_tokens": 250,

    "temperature": 0.2
}

print(config)

{'model_name': 'Qwen/Qwen2.5-3B-Instruct', 'max_new_tokens': 250, 'temperature': 0.2}


In [5]:
baseline_df = pd.read_csv(
    OUTPUT_DIR / "baseline_results.csv"
)

print(baseline_df.shape)

baseline_df.head()

(3, 8)


,instance_id,repo,problem_statement,base_commit,inference_time,input_tokens,output_tokens,response
0,pallets__flask-4045,pallets/flask,Raise error when blueprint name contains a dot...,d8c37f43724cd9fb0870f77877b7c4c7e38a19e0,19.40,107,300,### Analysis\n\n#### 1. Root Cause\n\nThe root...
1,pytest-dev__pytest-11143,pytest-dev/pytest,Rewrite fails when first expression of file is...,6995257cf470d2143ad1683824962de4071c0eb7,18.85,1888,300,### Analysis\n\n#### Root Cause\n\nThe error o...
2,astropy__astropy-12907,astropy/astropy,Modeling's `separability_matrix` does not comp...,d16bfe05a744909de4b27f5875fe0d4ed41ce607,16.47,382,300,### Analysis\n\n#### 1. Root Cause\n\nThe root...


In [6]:
MODEL_NAME = config["model_name"]

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    device_map="auto",

    torch_dtype=torch.float16
)

print("Model Ready")

Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model Ready


In [29]:
@dataclass
class Agent:

    name: str
    role: str
    system_prompt: str

    def run(self, task, context=""):

        prompt = f"""
You are {self.role}.

{self.system_prompt}

Repository:
{task["repo"]}

GitHub Issue:
{task["problem_statement"]}

Previous Context:
{context}

Provide only your assigned output.
"""

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        start = time.time()

        outputs = model.generate(

            **inputs,

            max_new_tokens=config["max_new_tokens"],

            temperature=config["temperature"],

            do_sample=False

        )

        elapsed = time.time() - start

        response = tokenizer.decode(

            outputs[0][inputs.input_ids.shape[-1]:],

            skip_special_tokens=True

        )

        return {
    "agent": self.name,
    "role": self.role,
    "prompt": prompt,
    "response": response,
    "time": round(elapsed, 2),
    "input_tokens": inputs.input_ids.shape[1],
    "output_tokens": outputs.shape[1] - inputs.input_ids.shape[1]
}

In [10]:
test_agent = Agent(

    name="Test",

    role="Software Engineer",

    system_prompt="""
Summarize the GitHub issue.

Identify the main software problem.

Do not propose a solution.
"""
)

sample_task = baseline_df.iloc[0]

result = test_agent.run(sample_task)

print(result["response"])

The main software problem identified in the GitHub issue is that raising an error when a blueprint name contains a dot is necessary due to the nesting capability of blueprints in Flask 1.0 and onwards. The issue notes that an error was already implemented for endpoint names, but it should also apply to blueprint names. ```plaintext
Main Software Problem:
Raise an error when a blueprint name contains a dot, to align with the requirement for nested blueprints in Flask versions 1.0 and above.
```


Task Planner

In [30]:
planner = Agent(

    name="Task Planner",

    role="Senior Software Engineering Manager",

    system_prompt="""
Your responsibility is to create a debugging plan.

Read the GitHub issue carefully.

Break the investigation into clear sequential steps.

Do NOT diagnose the bug.

Do NOT propose a solution.

Output only the investigation plan.
"""
)

print("Planner created")

Planner created


Repository Investigator

In [31]:
repository_agent = Agent(

    name="Repository Investigator",

    role="Senior Repository Maintainer",

    system_prompt="""
You specialize in understanding software repositories.

Using the GitHub issue and any previous planning context:

Identify:

- likely files

- modules

- classes

- functions

that should be inspected.

Explain WHY each component may be relevant.

Do not diagnose the bug.

Do not provide implementation details.
"""
)

print("Repository Agent created")

Repository Agent created


Root Cause Analyst

In [32]:
analysis_agent = Agent(

    name="Root Cause Analyst",

    role="Senior Debugging Engineer",

    system_prompt="""
Analyze the issue using:

- GitHub issue

- previous investigation

Identify the most probable root cause.

Explain your reasoning.

Avoid implementation details.
"""
)

print("Analysis Agent created")

Analysis Agent created


Solution Architect

In [33]:
solution_agent = Agent(

    name="Solution Architect",

    role="Principal Software Architect",

    system_prompt="""
Design a high-level implementation strategy.

Explain:

- what should change

- where it should change

- why it should change

Do not write code.

Do not generate a git patch.
"""
)

print("Solution Agent created")

Solution Agent created


Code Reviewer

In [34]:
review_agent = Agent(

    name="Code Reviewer",

    role="Senior Code Reviewer",

    system_prompt="""
Review the complete reasoning produced so far.

Evaluate:

- completeness

- consistency

- possible risks

Improve the final recommendation.

Produce the final software engineering recommendation.
"""
)

print("Reviewer created")

Reviewer created


Test Each Agent

In [16]:
sample_task = baseline_df.iloc[0]

In [17]:
planner_result = planner.run(sample_task)

print(planner_result["response"])

Investigation Plan:

1. Verify if the issue is reproducible on the latest version of Flask (1.x series).
2. Check if the issue occurs with different Python versions and operating systems.
3. Review the code changes introduced in Flask 1.0 that specifically address endpoint names.
4. Investigate if there are any specific examples or test cases in the Flask repository that cover blueprint names containing dots.
5. Examine the error message provided in the GitHub issue to understand the exact nature of the error being raised.
6. Search for related issues or pull requests in the Flask repository that might have addressed similar concerns.
7. Look for any documentation updates or changes that mention the handling of blueprint names containing dots.
8. Consult with other contributors or maintainers of the Flask project to gather insights and confirm the issue's severity and priority.
9. Create a minimal reproduction example to isolate the problem and ensure it is not caused by external facto

In [18]:
repository_result = repository_agent.run(

    sample_task,

    context=f"""
Previous Agent: Task Planner

Output:

{planner_result["response"]}
"""
)

print(repository_result["response"])

Investigation Plan:

1. Verify if the issue is reproducible on the latest version of Flask (1.x series).
   - Relevant because the issue description mentions "every dot is now significant since blueprints can be nested," indicating a change in behavior that may affect newer versions.

2. Check if the issue occurs with different Python versions and operating systems.
   - Relevant to ensure the issue is not isolated to a specific environment, affecting a broader user base.

3. Review the code changes introduced in Flask 1.0 that specifically address endpoint names.
   - Relevant as the issue description references an existing error for endpoint names, suggesting a related module or class might need additional checks.

4. Investigate if there are any specific examples or test cases in the Flask repository that cover blueprint names containing dots.
   - Relevant to identify if such tests already exist, which could help validate the presence of the issue.

5. Examine the error message pro

In [19]:
analysis_result = analysis_agent.run(

    sample_task,

    context=f"""
Task Planner

{planner_result["response"]}

Repository Investigator

{repository_result["response"]}
"""
)

print(analysis_result["response"])

The most probable root cause of the issue described in the GitHub issue is a bug in how blueprint names containing dots are handled in Flask. Given that the issue is related to the nesting of blueprints and the significance of dots, it suggests that there might be a misinterpretation or oversight in the code responsible for parsing or validating blueprint names. Specifically, the error seems to stem from a scenario where a blueprint name includes a dot, which is now treated as a separator rather than a regular character due to the nested structure of blueprints. This misinterpretation could lead to incorrect routing or validation logic, resulting in an error when attempting to use such a blueprint name. The absence of comprehensive tests covering this edge case further supports the likelihood of a coding oversight or incomplete handling of nested blueprint names. To confirm this hypothesis, a thorough review of the relevant code paths, especially those dealing with blueprint names and 

In [20]:
solution_result = solution_agent.run(

    sample_task,

    context=f"""
Planner

{planner_result["response"]}

Repository

{repository_result["response"]}

Analysis

{analysis_result["response"]}
"""
)

print(solution_result["response"])

The most probable root cause of the issue described in the GitHub issue is a bug in how blueprint names containing dots are handled in Flask. Given that the issue is related to the nesting of blueprints and the significance of dots, it suggests that there might be a misinterpretation or oversight in the code responsible for parsing or validating blueprint names. Specifically, the error seems to stem from a scenario where a blueprint name includes a dot, which is now treated as a separator rather than a regular character due to the nested structure of blueprints. This misinterpretation could lead to incorrect routing or validation logic, resulting in an error when attempting to use such a blueprint name. The absence of comprehensive tests covering this edge case further supports the likelihood of a coding oversight or incomplete handling of nested blueprint names. To confirm this hypothesis, a thorough review of the relevant code paths, especially those dealing with blueprint names and 

In [21]:
review_result = review_agent.run(

    sample_task,

    context=f"""
Planner

{planner_result["response"]}

Repository

{repository_result["response"]}

Analysis

{analysis_result["response"]}

Solution

{solution_result["response"]}
"""
)

print(review_result["response"])

Final Software Engineering Recommendation:

**Recommendation:**

Given the analysis and the identified root cause, the most appropriate action is to raise a new issue in the Flask repository to address the handling of blueprint names containing dots. This issue should include a detailed explanation of the problem, including the error message, the steps to reproduce the issue, and a proposed solution. The proposed solution should involve adding robust validation and parsing logic to handle blueprint names containing dots correctly. Additionally, the team should consider adding comprehensive tests to cover this edge case, ensuring that such issues do not arise in the future.

**Justification:**

- **Completeness:** The recommendation addresses the root cause of the issue, proposes a clear course of action, and includes a detailed plan for verification and testing.
- **Consistency:** The recommendation aligns with best practices in software development, including thorough testing and docu

GitHub Issue
      │
      ▼
Task Planner
      │
      ▼
Repository Investigator
      │
      ▼
Root Cause Analyst
      │
      ▼
Solution Architect
      │
      ▼
Code Reviewer
      │
      ▼
Final Recommendation

↓

Save Complete Trace

Create AgentState

In [9]:
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class AgentState:

    instance_id: str
    repo: str
    problem_statement: str

    steps: list = field(default_factory=list)

    final_response: str = ""

    def add_step(self, result):

        step = {

            "step": len(self.steps) + 1,

            "timestamp": datetime.utcnow().isoformat(),

            **result

        }

        self.steps.append(step)

    def context(self):

        if not self.steps:
            return ""

        context = ""

        for step in self.steps:

            context += f"""
=========================================
Step: {step['step']}

Agent:
{step['agent']}

Role:
{step['role']}

Response:
{step['response']}

"""

        return context

In [3]:
def run_pipeline(task):

    # -----------------------------------------
    # Initialize workflow state
    # -----------------------------------------
    state = AgentState(

        instance_id=task["instance_id"],

        repo=task["repo"],

        problem_statement=task["problem_statement"]

    )

    # -----------------------------------------
    # Ordered execution of agents
    # -----------------------------------------
    agents = [

        planner,

        repository_agent,

        analysis_agent,

        solution_agent,

        review_agent

    ]

    # -----------------------------------------
    # Execute workflow
    # -----------------------------------------
    for agent in agents:

        result = agent.run(

            task,

            context=state.context()

        )

        state.add_step(result)

    # -----------------------------------------
    # Final response comes from reviewer
    # -----------------------------------------
    state.final_response = state.steps[-1]["response"]

    # -----------------------------------------
    # Build complete reasoning trace
    # (Used later for fine-tuning)
    # -----------------------------------------
    state.trace_text = "\n\n".join(

        [

            f"""Step {step['step']} - {step['agent']}

Role:
{step['role']}

Response:
{step['response']}"""

            for step in state.steps

        ]

    )

    # -----------------------------------------
    # Statistics
    # -----------------------------------------
    state.total_time = round(

        sum(step["time"] for step in state.steps),

        2

    )

    state.total_input_tokens = sum(

        step["input_tokens"]

        for step in state.steps

    )

    state.total_output_tokens = sum(

        step["output_tokens"]

        for step in state.steps

    )

    return state

In [10]:
def run_pipeline(task):

    trace = []

    context = ""

    agents = [
        planner,
        repository_agent,
        analysis_agent,
        solution_agent,
        review_agent
    ]

    for agent in agents:

        print(f"Running {agent.name}...")

        result = agent.run(
            task,
            context=context
        )

        trace.append(result)

        context += f"""

Agent: {result['agent']}

Response:
{result['response']}

"""

    return trace

In [11]:
agent_results = []

agent_traces = []

NUM_TASKS = 1   # Keep 1 while testing

for idx, (_, task) in enumerate(
    baseline_df.head(NUM_TASKS).iterrows(),
    start=1
):

    print("=" * 80)
    print(f"Task {idx}/{NUM_TASKS}")
    print(task["instance_id"])

    trace = run_pipeline(task)

    final_response = trace[-1]["response"]

    total_time = sum(step["time"] for step in trace)

    total_input_tokens = sum(step["input_tokens"] for step in trace)

    total_output_tokens = sum(step["output_tokens"] for step in trace)

    agent_results.append({

        "instance_id": task["instance_id"],

        "repo": task["repo"],

        "problem_statement": task["problem_statement"],

        "final_response": final_response,

        "total_time": round(total_time, 2),

        "input_tokens": total_input_tokens,

        "output_tokens": total_output_tokens

    })

    agent_traces.append({

        "instance_id": task["instance_id"],

        "repo": task["repo"],

        "problem_statement": task["problem_statement"],

        "steps": trace

    })

    print(f"Completed in {total_time:.2f} sec")

Task 1/1
pallets__flask-4045
Running Task Planner...
Running Repository Investigator...
Running Root Cause Analyst...
Running Solution Architect...
Running Code Reviewer...
Completed in 1971.20 sec


In [12]:
results_df = pd.DataFrame(agent_results)

results_df.to_csv(
    OUTPUT_DIR / "agent_results.csv",
    index=False
)

results_df.to_json(
    OUTPUT_DIR / "agent_results.json",
    orient="records",
    indent=2
)

print("Saved agent results.")

Saved agent results.


In [13]:
import json

with open(
    OUTPUT_DIR / "agent_traces.json",
    "w"
) as f:

    json.dump(
        agent_traces,
        f,
        indent=2
    )

print("Saved agent traces.")

Saved agent traces.


In [14]:
experiment = {

    "experiment": "multi_agent",

    "model": MODEL_NAME,

    "agents": 5,

    "tasks": NUM_TASKS,

    "workflow": [
        "Task Planner",
        "Repository Investigator",
        "Root Cause Analyst",
        "Solution Architect",
        "Code Reviewer"
    ]

}

with open(
    OUTPUT_DIR / "agent_experiment.json",
    "w"
) as f:

    json.dump(
        experiment,
        f,
        indent=4
    )

print("Experiment saved.")

Experiment saved.


In [15]:
sample = agent_traces[0]

print("=" * 80)

for step in sample["steps"]:

    print(f"\n### {step['agent']} ###\n")

    print(step["response"])

print("=" * 80)


### Task Planner ###

Investigation Plan:

1. Verify if the issue is reproducible on the latest version of Flask (1.x series).
2. Check if the issue occurs with different Python versions and operating systems.
3. Review the code changes introduced in Flask 1.0 that specifically address endpoint names.
4. Investigate if there are any specific configurations or settings that might affect the behavior when using blueprints with dots in their names.
5. Examine the error message provided in the GitHub issue to understand the exact nature of the error being raised.
6. Search for similar issues or discussions related to this problem

### Repository Investigator ###

Response:
1. Likely Files: flask/app.py, flask/blueprints.py, flask/helpers.py
   - Reason: These files contain the core logic for handling blueprints and endpoints in Flask.

2. Modules: flask.blueprints, flask.helpers
   - Reason: These modules encapsulate the functionality related to blueprint management and utility functions,

Notebook 04 - Phase 4: Analysis & Comparison
Objective

Compare:

Baseline SLM
Multi-Agent Workflow

using measurable metrics.

In [16]:
baseline_df = pd.read_csv(
    OUTPUT_DIR / "baseline_results.csv"
)

agent_df = pd.read_csv(
    OUTPUT_DIR / "agent_results.csv"
)

print("Baseline:", baseline_df.shape)
print("Multi-Agent:", agent_df.shape)

Baseline: (3, 8)
Multi-Agent: (1, 7)


In [18]:
print(baseline_df.columns.tolist())

['instance_id', 'repo', 'problem_statement', 'base_commit', 'inference_time', 'input_tokens', 'output_tokens', 'response']


In [19]:
print(agent_df.columns.tolist())

['instance_id', 'repo', 'problem_statement', 'final_response', 'total_time', 'input_tokens', 'output_tokens']


In [20]:
comparison = {

    "Baseline Tasks": len(baseline_df),

    "Agent Tasks": len(agent_df),

    "Baseline Avg Time (sec)": round(
        baseline_df["inference_time"].mean(),
        2
    ),

    "Agent Avg Time (sec)": round(
        agent_df["total_time"].mean(),
        2
    ),

    "Baseline Avg Input Tokens": round(
        baseline_df["input_tokens"].mean(),
        2
    ),

    "Agent Avg Input Tokens": round(
        agent_df["input_tokens"].mean(),
        2
    ),

    "Baseline Avg Output Tokens": round(
        baseline_df["output_tokens"].mean(),
        2
    ),

    "Agent Avg Output Tokens": round(
        agent_df["output_tokens"].mean(),
        2
    )

}

comparison

{'Baseline Tasks': 3,
 'Agent Tasks': 1,
 'Baseline Avg Time (sec)': np.float64(18.24),
 'Agent Avg Time (sec)': np.float64(1971.2),
 'Baseline Avg Input Tokens': np.float64(792.33),
 'Agent Avg Input Tokens': np.float64(1873.0),
 'Baseline Avg Output Tokens': np.float64(300.0),
 'Agent Avg Output Tokens': np.float64(600.0)}

In [21]:
comparison_df = pd.DataFrame({

    "Metric": [

        "Tasks",

        "Average Inference Time (sec)",

        "Average Input Tokens",

        "Average Output Tokens"

    ],

    "Baseline SLM": [

        comparison["Baseline Tasks"],

        comparison["Baseline Avg Time (sec)"],

        comparison["Baseline Avg Input Tokens"],

        comparison["Baseline Avg Output Tokens"]

    ],

    "Multi-Agent Workflow": [

        comparison["Agent Tasks"],

        comparison["Agent Avg Time (sec)"],

        comparison["Agent Avg Input Tokens"],

        comparison["Agent Avg Output Tokens"]

    ]

})

comparison_df

,Metric,Baseline SLM,Multi-Agent Workflow
0,Tasks,3.00,1.0
1,Average Inference Time (sec),18.24,1971.2
2,Average Input Tokens,792.33,1873.0
3,Average Output Tokens,300.00,600.0


In [22]:
comparison_df.to_csv(
    OUTPUT_DIR / "comparison_metrics.csv",
    index=False
)

comparison_df.to_json(
    OUTPUT_DIR / "comparison_metrics.json",
    orient="records",
    indent=2
)

print("Comparison metrics saved.")

Comparison metrics saved.


In [23]:
analysis = {

    "observations": [

        "The multi-agent workflow executes multiple specialized agents sequentially, increasing total inference time.",

        "Input token count is higher because each agent receives reasoning generated by previous agents.",

        "Output token count is larger because the workflow produces intermediate reasoning before generating the final recommendation.",

        "The generated reasoning traces will be used to fine-tune a local Small Language Model in subsequent notebooks."

    ]

}

with open(
    OUTPUT_DIR / "analysis_summary.json",
    "w"
) as f:

    json.dump(
        analysis,
        f,
        indent=4
    )

print("Analysis summary saved.")

Analysis summary saved.


In [24]:
print("=" * 80)
print("MULTI-AGENT WORKFLOW ANALYSIS")
print("=" * 80)

print(f"Baseline Tasks              : {comparison['Baseline Tasks']}")
print(f"Multi-Agent Tasks           : {comparison['Agent Tasks']}")

print()

print(f"Baseline Avg Time (sec)     : {comparison['Baseline Avg Time (sec)']}")
print(f"Multi-Agent Avg Time (sec)  : {comparison['Agent Avg Time (sec)']}")

print()

print(f"Baseline Avg Input Tokens   : {comparison['Baseline Avg Input Tokens']}")
print(f"Multi-Agent Input Tokens    : {comparison['Agent Avg Input Tokens']}")

print()

print(f"Baseline Avg Output Tokens  : {comparison['Baseline Avg Output Tokens']}")
print(f"Multi-Agent Output Tokens   : {comparison['Agent Avg Output Tokens']}")

print()
print("Notebook 04 Completed Successfully")
print("=" * 80)

MULTI-AGENT WORKFLOW ANALYSIS
Baseline Tasks              : 3
Multi-Agent Tasks           : 1

Baseline Avg Time (sec)     : 18.24
Multi-Agent Avg Time (sec)  : 1971.2

Baseline Avg Input Tokens   : 792.33
Multi-Agent Input Tokens    : 1873.0

Baseline Avg Output Tokens  : 300.0
Multi-Agent Output Tokens   : 600.0

Notebook 04 Completed Successfully
